# 为分类任务微调表示模型

在第4章中，我们使用预训练模型完成了文本分类任务，当时完全保留了预训练模型的原始参数，没有进行任何调整。这种处理方式自然引出一个疑问：如果对这些模型进行微调，结果会如何？

实践证明，当具备充足数据时，微调通常能获得最佳性能。本章将系统讲解BERT模型的多种微调方法和应用场景。11.1节将演示微调分类模型的一般流程；11.2节将介绍SetFit方法——一种利用少量训练样本高效微调高性能模型的创新方案；11.3节将探索预训练模型的持续训练策略；11.4节将深入探讨词元级别的分类任务。

本章将重点探讨非生成式任务，生成模型的相关内容将留待第12章专门论述。

## 11.1 监督分类

回顾第4章，我们利用预训练的表示模型处理监督分类任务时，采用的模型可分为两类：专门用于情感分析的特定任务模型和用于创建嵌入向量的嵌入模型，如图11-1所示。

<div align="center">
    <img src="./picture/11-1.jpg">
    <p>图11-1：第4章使用预训练模型进行分类时保持模型权重冻结状态</p>
</div>


为充分展示预训练模型的分类潜力，两种模型均保持参数冻结（不可训练状态）​。嵌入模型通过独立可训练的分类头（分类器）来实现影评情感预测。

本节将采用类似的架构，但允许模型参数与分类头在训练过程中同步更新。如图11-2所示，我们不再使用嵌入模型，而是直接微调预训练的BERT模型，以构建类似于我们在第4章中使用的特定任务模型。相较于嵌入模型方案，该方法将表示模型与分类头作为一个整体架构进行端到端微调。

<div align="center">
    <img src="./picture/11-2.jpg">
    <p>图11-2：相较于“冻结”架构方案，新方法实现了BERT模型与分类头的联合训练。反向传播过程将从分类头开始，逐层贯穿整个BERT模型</p>
</div>

为此，我们不再冻结模型参数，而是允许其保持可训练状态，在训练过程中动态更新。如图11-3所示，我们将采用一个预训练的BERT模型作为基础架构，并为其添加一个神经网络分类头，二者都将通过微调来适应具体分类任务的需求。

<div align="center">
    <img src="./picture/11-3.jpg">
    <p>图11-3：面向特定任务的模型架构示意图。该架构由预训练的表示模型（如BERT）和针对任务设计的附加分类头组成</p>
</div>

在实际训练过程中，预训练的BERT模型与分类头将共同进行参数更新。二者不再是彼此独立的训练组件，而是通过参数联动实现协同优化，从而生成更具判别力的语义表征。

### 11.1.1 微调预训练的BERT模型

我们将沿用第4章使用的烂番茄数据集进行模型的微调。该数据集包含来自影评网站的5331条正面影评与5331条负面影评。

In [2]:
from datasets import load_dataset
# 准备数据并进行数据划分
tomatoes = load_dataset("cornell-movie-review-data/rotten_tomatoes")
train_data, test_data = tomatoes["train"], tomatoes["test"]

README.md:   0%|          | 0.00/7.46k [00:00<?, ?B/s]

d:\study\python\AiAgent\图解大模型\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\admin\.cache\huggingface\hub\datasets--cornell-movie-review-data--rotten_tomatoes. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\study\python\AiAgent\图解大模型\.venv\Lib\site-packages\huggingface_hub\file_download.py:15

train.parquet: reconstructing file:   0%|          |  0.00B /  699kB            

train.parquet: downloading bytes:           |  0.00B            

validation.parquet: reconstructing file:   0%|          |  0.00B / 90.0kB            

...

test.parquet: reconstructing file:   0%|          |  0.00B / 92.2kB            

test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

分类任务的第一步是选择合适的基础模型。在本示例中，我们选用bert-base-cased模型，该模型已在英文维基百科和一个由未出版书籍组成的大型数据集上完成了预训练。

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
# 加载模型和分词器
model_id = "../models/bert-base-cased"
model = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=2
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '../models/bert-base-cased'. Use `repo_type` argument if needed.

接下来，我们将对数据进行分词处理：

from transformers import DataCollatorWithPadding
# 对批次中的序列进行填充，使其长度与最长序列一致
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
def preprocess_function(examples):
    """对输入数据进行分词处理"""
    return tokenizer(examples["text"], truncation=True)
# 对训练数据和测试数据进行分词处理
tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_test = test_data.map(preprocess_function, batched=True)

在构建训练器(Trainer)之前，我们需要准备一个专用的数据收集器(DataCollator)。数据收集器的主要功能是构建数据批次，同时支持数据增强技术的应用。

如第9章所述的分词处理流程，我们需要对输入文本进行填充操作，以确保生成大小一致的表示。这正是DataCollatorWithPadding工具的核心作用。

完整的训练示例还需要明确定义若干评估指标：

In [ ]:
import numpy as np
from datasets import load_metric
def compute_metrics(eval_pred):
   """计算F1分数"""
   logits, labels = eval_pred
   predictions = np.argmax(logits, axis=-1)
   load_f1 = load_metric("f1")
   f1 = load_f1.compute(predictions=predictions, references=labels)["f1"]
   return {"f1": f1}

通过compute_metrics函数，我们可以定义任意数量的评估指标，这些指标在训练过程中被实时显示或记录保存。该功能对于监测模型训练尤为重要，因为它能帮助我们及时检测过拟合现象。

下面我们初始化Trainer对象：

In [ ]:
from transformers import TrainingArguments, Trainer
# 用于参数调优的训练参数
training_args = TrainingArguments(
   "model",
   learning_rate=2e-5,
   per_device_train_batch_size=16,
   per_device_eval_batch_size=16,
   num_train_epochs=1,
   weight_decay=0.01,
   save_strategy="epoch",
   report_to="none"
)
# 执行训练过程的Trainer
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   tokenizer=tokenizer,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)

TrainingArguments类定义了模型调优所需的超参数，例如学习率与训练轮次(epochs)，而Trainer则负责具体执行训练流程。

最后，我们训练模型并进行评估：

In [ ]:
trainer.evaluate()

该模型的F1分数为0.85，这明显高于第4章中特定任务模型的F1分数(0.80)。这表明自行微调模型可能比直接使用预训练模型更具优势，整个训练过程仅需数分钟即可完成。

### 11.1.2 冻结层

为进一步展示训练整个网络的重要性，以下示例将演示如何使用Hugging FaceTransformers冻结网络的部分层。

本例将冻结BERT模型的主体结构，仅允许对分类头进行更新。这种对比实验设计极具参考价值，因为我们维持其他条件不变，仅对指定层进行冻结操作。

首先，我们需要重新初始化模型以重置参数状态：

In [ ]:
# 加载模型和分词器
model = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=2
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

我们预训练的BERT模型包含多个可冻结层。通 过检查这些层，我们能够深入理解网络结构，并确定可能需要冻结的部分。

In [ ]:
# 打印层的名称
for name, param in model.named_parameters():
    print(name)

该模型由12个（编号为0~11）编码器块构成，每个块均包含注意力头、前馈神经网络以及层归一化组件。图11-4详细地展示了该架构，并标注了所有可被冻结的模块。在基础架构上方还配置了独立的分类头模块。

<div align="center">
    <img src="./picture/11-4.jpg">
    <p>通过选择性冻结特定网络层，我们能在提升计算效率的同时，仍使模型主体通过分类任务进行参数更新。在典型配置中，建议将冻结层与可训练层进行交替排列。</p>
</div>

通过选择性冻结特定网络层，我们能在提升计算效率的同时，仍使模型主体通过分类任务进行参数更新。在典型配置中，建议将冻结层与可训练层进行交替排列。

如第4章所述，我们将冻结除分类头之外的所有结构：

In [ ]:
for name, param in model.named_parameters():
     # 可训练的分类头
     if name.startswith("classifier"):
        param.requires_grad = True
      # 冻结其他所有结构
     else:
        param.requires_grad = False

如图11-5所示，我们冻结了除前馈神经网络（分类头）之外的所有结构。

<div align="center">
    <img src="./picture/11-5.jpg">
    <p>图11-5：我们冻结所有编码器块和嵌入层，使得BERT模型在微调过程中不会学习新的表示</p>
</div>

在成功冻结除分类头之外的所有结构后，即可继续进行模型训练：

In [ ]:
from transformers import TrainingArguments, Trainer
# 执行训练过程的Trainer
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   tokenizer=tokenizer,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)
trainer.train()

你可能已经注意到训练速度明显提升。这是因为我们仅对分类头进行训练，相比微调整个模型，这能显著提升训练速度。

In [ ]:
trainer.evaluate()

该模型的F1分数仅为0.63，相较于最初的分数(0.85)显著下降。与其冻结几乎所有层，不如如图11-6所示，仅冻结前10个编码器块，然后观察这种调整对模型性能的影响。这种策略的显著优势在于：既能有效降低计算开销，又能使梯度更新通过部分预训练模型进行反向传播。

<div align="center">
    <img src="./picture/11-6.jpg">
    <p>图11-6：我们仅冻结BERT模型的前10个编码器块，其余部分均为可训练的并将进行微调</p>
</div>

In [ ]:
# 加载模型
model_id = "../models/bert-base-cased"
model = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=2
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
# 编码器块11从索引165开始
# 我们冻结该块之前的所有结构
for index, (name, param) in enumerate(model.named_parameters()):
    if index < 165:
        param.requires_grad = False
# 执行训练过程的Trainer
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   tokenizer=tokenizer,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)
trainer.train()

在完成训练后，我们对模型性能进行全面评估：

In [ ]:
trainer.evaluate()
{'eval_loss': 0.40812647342681885,
 'eval_f1': 0.8,
 'eval_runtime': 3.7125,
 'eval_samples_per_second': 287.137,
 'eval_steps_per_second': 18.047,
 'epoch': 1.0}

该模型的F1分数为0.8，这比我们之前冻结全部层时取得的0.63有了显著提升。这一结果表明，尽管我们通常会优先考虑训练尽可能多的层，但在计算资源受限的情况下，仅训练部分层仍能取得可接受的效果。

为进一步验证这种效果，我们延续之前的实验范式，逐步冻结编码器块并实施微调操作，同时评估模型性能。如图11-7所示，仅需训练前5个编码器块（垂直虚线标注位置）即可获得接近完整训练所有编码器块的性能表现。

<div align="center">
    <img src="./picture/11-7.jpg">
    <p>图11-7：冻结特定编码器块对模型性能的影响。训练更多模块带来的性能提升会快速趋于平缓</p>
</div>

随着训练轮次的增加，冻结与不冻结策略在训练耗时和资源消耗方面的差异将越发显著。因此建议尝试不同的冻结配置方案，在性能与效率之间找到最优平衡点。

## 11.2 少样本分类

少样本分类作为监督学习分类的一种特殊技术，能使分类器仅通过少量标注样本即可学习并识别不同的目标标签。对于缺乏现成的标注数据的分类任务而言，这种技术尤为实用。这种技术的核心思想是通过为每个类别精心标注少量高质量的数据点来完成模型训练。图11-8直观地展示了基于少量标注数据进行模型训练的方法。

<div align="center">
    <img src="./picture/11-8.jpg">
    <p>图11-8：少样本分类仅需少量标注数据即可完成模型训练</p>
</div>

### 11.2.1 SetFit：少样本场景下的高效微调方案

为实现少样本文本分类，我们采用名为SetFit的高效框架。该框架基于sentence-transformers架构构建，能够在训练过程中动态优化文本表示质量。仅需少量标注样本，SetFit框架的表现即可媲美前文示例中基于大规模标注数据集微调的BERT模型。

SetFit的核心算法流程包含三个关键阶段。

· 采样训练数据。通过对标注数据的类内与类间样本选择，生成包含正例（相似）与负例（不相似）的句子对。

· 微调嵌入模型。利用生成的训练数据，对预训练的嵌入模型进行微调。

· 训练分类器。在优化后的嵌入模型的基础上构建分类头，并使用之前生成的训练数据对其进行训练。

在微调嵌入模型之前，我们需要生成训练数据。该模型假设训练数据为包含正例（相似）和负例（不相似）的句子对。然而，当我们处理分类任务时，原始输入数据通常并不具备此类标签。

以图11-9所示的训练数据集为例，该数据集将文本分为两类：关于编程语言的文本和关于宠物的文本。

<div align="center">
    <img src="./picture/11-9.jpg">
    <p>图11-9：两个类别的数据：关于编程语言的文本和关于宠物的文本</p>
</div>

如步骤1所示（见图11-10）​，SetFit通过类内和类间的选择机制生成所需数据。例如，在有16个运动类句子的情况下，我们可创建16×(16-1)/2=120个正例句子对，通过跨类别采样创建负例句子对。

<div align="center">
    <img src="./picture/11-10.jpg">
    <p>图11-10：步骤1，采样训练数据。我们假设同一类别内的句子具有相似性并构建正例对，不同类别中的句子则构成负例对</p>
</div>

步骤2的核心是利用生成的句子对微调嵌入模型。具体而言，我们采用对比学习方法对预训练的BERT模型进行微调。正如第10章所述，对比学习可通过正例（相似的句子对）和负例（不相似的句子对）的对比训练，获得精准的句子嵌入。

由于步骤1已生成所需的句子对，因此我们可以直接使用这些数据微调SentenceTransformer模型。虽然对比学习的概念前文已有阐述，但为便于理解，图11-11仍对此方法进行了可视化说明。

<div align="center">
    <img src="./picture/11-11.jpg">
    <p>图11-11：步骤2，微调SentenceTransformer模型。通过对比学习，从正例和负例句子对中学习嵌入向量</p>
</div>

微调嵌入模型的目标是使其生成适应分类任务的嵌入向量。通过微调嵌入模型，类别的相关性及其相对关系被提炼至嵌入向量中。

在步骤3中，我们为所有句子生成嵌入向量，并将这些嵌入向量作为分类器的输入。我们可以使用微调后的SentenceTransformer模型将句子转化为嵌入向量，作为特征使用。分类器通过学习经过优化的嵌入向量，实现对未知句子的准确预测。步骤3的流程如图11-12所示。

通过整合所有步骤，我们最终构建出一个高效且优雅的流程，能够在每个类别仅有少量标注数据的情况下完成分类任务。该流程巧妙地利用了现有的标注数据资源，尽管这些数据与目标任务并不完全匹配。图11-13整合了三个步骤，完整展示了SetFit在少样本场景下的全流程架构。

<div align="center">
    <img src="./picture/11-12.jpg">
    <p>图11-12：步骤3，训练分类器。可选择任意scikit-learn提供的分类模型或分类头作为分类器</p>
</div>

<div align="center">
    <img src="./picture/11-13.jpg">
    <p>图11-13：SetFit的三个核心阶段</p>
</div>

SetFit的具体实现包含三个核心阶段：首先，通过类内和类间选择生成句子对；其次，利用这些句子对微调预训练的SentenceTransformer模型；最后，采用微调后的模型生成句子嵌入，并基于此训练分类器。

### 11.2.2 少样本分类的微调

在前述示例中，我们在包含约8500条影评的数据集上进行训练。然而，由于这是一个少样本学习场景，我们仅需从每个类别中抽取16个样本。对于二分类任务，这意味着仅使用32个句子即可完成训练。相较于常规训练所需的8500条影评，样本规模缩减了3个数量级，差异尤为显著。

In [ ]:
from setfit import sample_dataset
# 我们通过从每个类别中抽取16个样本来模拟少样本设置
sampled_train_data = sample_dataset(tomatoes["train"], num_samples=16)

在完成数据采样后，我们将选取预训练的SentenceTransformer模型进行微调。官方文档提供了预训练的SentenceTransformer模型的详细列表，本次选用的是sentence-transformers/all-mpnet-base-v2。该模型在MTEB排行榜中名列前茅，MTEB排行榜全面展示了嵌入模型在语义搜索、聚类分析等多类任务中的性能表现。

In [ ]:
from setfit import SetFitModel
# 加载预训练的SentenceTransformer模型
model = SetFitModel.from_pretrained("../models/all-mpnet-base-v2")

加载预训练的SentenceTransformer模型后，我们即可创建训练器SetFitTrainer了。该训练器默认采用逻辑回归模型作为待训练的分类器。

与使用Hugging Face Transformers时的操作类似，我们可以通过训练器来配置相关参数。例如将num_epochs参数设为3，这意味着对比学习过程将完成3轮迭代训练。

In [ ]:
from setfit import TrainingArguments as SetFitTrainingArguments
from setfit import Trainer as SetFitTrainer
# 定义训练参数
args = SetFitTrainingArguments(
    num_epochs=3, # 用于对比学习的轮次
    num_iterations=20 # 要生成的文本对数量
)
args.eval_strategy = args.evaluation_strategy
# 创建训练器
trainer = SetFitTrainer(
    model=model,
    args=args,
    train_dataset=sampled_train_data,
    eval_dataset=test_data,
    metric="f1"
)

我们只需调用train函数即可启动训练循环。执行该操作后，可得到如下输出：

In [ ]:
# 训练循环
trainer.train()

经过微调的SentenceTransformer模型共生成了1280个句子对。默认情况下，系统会为数据集中的每个样本生成20个句子对组合，即20×32=640个基础样本。由于每个正例对和负例对都需要进行双向扩展，因此实际数量为640×2=1280个句子对。考虑到最初我们仅有32个标注句子，能够生成1280个句子对已经非常出色！

当未明确定义分类头时，系统默认采用逻辑回归模型。若需自定义分类头，可以在SetFitTrainer中通过指定以下模型来实现。

In [ ]:
# 从Hub加载SetFit模型
model = SetFitModel.from_pretrained(
    "sentence-transformers/all-mpnet-base-v2",
    use_differentiable_head=True,
    head_params={"out_features": num_classes},
)
# 创建训练器
trainer = SetFitTrainer(
    model=model,
    ...
)

此处的num_classes表示我们需要预测的类别总数。

接下来，我们将对模型进行评估以考察其性能表现：

In [ ]:
# 在测试数据上评估模型
trainer.evaluate()

仅用32个标注句子，F1分数就达到了0.84。考虑到模型仅在原始数据的一小部分上进行训练，这个结果堪称惊艳！值得注意的是，在第4章中，我们也获得了相同的性能表现，但当时是在完整数据的嵌入向量上训练逻辑回归模型的。这一流程充分体现了投入时间标注少量样本的巨大潜力。

SetFit不仅能胜任少样本分类任务，还适用于完全无标注的零样本分类场景。SetFit的工作原理是，通过标注名称生成合成样本来模拟分类任务，随后在这些样本上训练SetFit模型。例如，当目标标签为happy和sad时，系统可能自动生成类似The example is happy和Thisexample is sad的合成句子。

## 11.3 基于掩码语言建模的继续预训练

在前述示例中，我们主要采用预训练模型并通过微调来完成分类任务。整个过程可分为两个步骤：首先对模型进行预训练（已完成的基础工作）​，然后针对特定任务进行微调。图11-14直观地展示了这一流程。

<div align="center">
    <img src="./picture/11-14.jpg">
    <p>图11-14：要在目标任务（如分类）上微调模型，我们可以从预训练BERT模型开始，或使用预训练好的BERT模型</p>
</div>

这种两步法虽被广泛应用，但在处理特定领域数据时具有一定的局限性。预训练模型通常基于非常通用的数据（如维基百科内容）进行训练，可能无法充分适应特定领域的专业词汇。

为此，我们可以在两个步骤之间增加继续预训练(continued pretraining)环节。具体而言，就是在预训练好的BERT模型的基础上，继续使用特定领域的数据实施掩码语言建模(MLM)训练。比如从一个通用的BERT模型优化到专门用于医学领域的BioBERT模型，再微调至用于药物分类的BioBERT模型。

这将更新子词表示，使其更好地适应特定领域的词汇。如图11-15所示，新增的继续预训练环节通过领域数据上的MLM任务优化模型。实践证明，在预训练的BERT模型上继续进行预训练能显著提升模型在分类任务中的性能，是微调过程中极具价值的增强手段。

<div align="center">
    <img src="./picture/11-15.jpg">
    <p>图11-15：区别于两步法，我们可以在针对目标任务进行微调之前对预训练模型进行继续预训练。需注意的是，在步骤1中掩码被填充为抽象概念，而在步骤2中掩码被填充为与电影相关的具体概念</p>
</div>

我们无须从头开始预训练整个模型，只需在对其进行微调以用于分类任务之前简单地进行继续预训练。这种方法还能帮助模型更好地适应特定领域，甚至是特定组织的专业术语。图11-16进一步展示了企业可能采用的模型演进路径。

<div align="center">
    <img src="./picture/11-16.jpg">
    <p>图11-16：针对特定用例的三步法示意图</p>
</div>

在本示例中，我们将演示如何实施步骤2——对预训练好的BERT模型进行继续预训练。我们使用与之前相同的数据集，即烂番茄影评数据集。

首先加载我们迄今一直使用的bert-base-cased模型，并为其配置掩码语言建模任务：

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
# 加载掩码语言建模(MLM)模型
model = AutoModelForMaskedLM.from_pretrained("../models/bert-base-cased")
tokenizer = AutoTokenizer.from_pretrained("../models/bert-base-cased")


此前，我们使用过能够动态填充输入内容的DataCollatorWithPadding。

本例将采用能够执行词元的掩码操作的数据整理器。掩码操作通常有两种实现方式：词元掩码和整词掩码。当使用词元掩码时，系统会随机掩码句子中15%的独立词元，这可能导致某个单词的部分字符被掩码。为了实现对整个单词的掩码，我们可以使用整词掩码，如图11-17所示。

<div align="center">
    <img src="./picture/11-17.jpg">
    <p>图11-17：随机掩码词元的不同方法</p>
</div>

通常，预测整个单词比预测单个词元更具挑战性，这种设计会促使模型在训练过程中学习更准确和精细的语义表示，从而提升性能。但需注意的是，这种方法需要更长的收敛时间。为加快训练速度，本示例将采用DataCollatorForLanguageModeling实现词元掩码。若需实施整词掩码，只需将数据整理器替换为DataCollatorForWholeWordMask即可。另外，我们将掩码概率（mlm_probability参数）设置为0.15，即在每个句子中随机选择15%的词元进行掩码处理。

In [ ]:
from transformers import DataCollatorForLanguageModeling
# 掩码词元
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

接下来，我们将创建用于执行掩码语言建模任务的Trainer，并指定下列参数配置：

In [ ]:
# 用于参数调优的训练参数
training_args = TrainingArguments(
   "model",
   learning_rate=2e-5,
   per_device_train_batch_size=16,
   per_device_eval_batch_size=16,
   num_train_epochs=10,
   weight_decay=0.01,
   save_strategy="epoch",
   report_to="none"
)
# 初始化Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator
)

这里有几个值得注意的关键参数。我们以20个训练周期进行训练，同时保持任务规模精简。建议尝试调整学习率与权重衰减系数，观察其对模型微调效果的影响。

在启动训练循环前，我们预先保存预训练的分词器。由于分词器参数在训练过程中不会更新，因此训练完成后无须重复保存。但值得注意的是，在继续预训练流程结束后，需完整保存更新后的模型状态。

In [ ]:
# 保存预训练的分词器
tokenizer.save_pretrained("mlm")
# 训练模型
trainer.train()
# 保存更新后的模型
model.save_pretrained("mlm")

这样，我们便在mlm文件夹中得到了更新后的模型。为评估其性能，我们通常会在各类任务上对模型进行微调。但就当前阶段而言，我们可以通过运行若干掩码任务来检验模型是否从继续训练中获得了知识提升。

我们通过在继续预训练之前加载预训练模型来完成这一操作。以句子“What a horrible[MASK]!”为例，模型将对“​[MASK]​”位置的单词进行预测。

In [ ]:
from transformers import pipeline
# 加载并创建预测
mask_filler = pipeline("fill-mask", model="bert-base-cased")
preds = mask_filler("What a horrible [MASK]!")
# 打印结果
for pred in preds:
    print(f">>> {pred["sequence"]}")

输出中包含idea、dream和day等单词，这些预测结果都非常合理。接下来，我们看看更新后的模型会做出怎样的预测。

In [ ]:
# 加载并创建预测
mask_filler = pipeline("fill-mask", model="mlm")
preds = mask_filler("What a horrible [MASK]!")
# 打印结果
for pred in preds:
    print(f">>> {pred["sequence"]}")

这次输出中包含movie、film、mess等单词，相较于预训练模型，该模型更贴合我们输入的数据特征。

下一步便是在本章开头提到的分类任务上对这个模型进行微调，只需按照以下方式加载模型即可开始：

In [ ]:
from transformers import AutoModelForSequenceClassification
# 用于分类的微调
model = AutoModelForSequenceClassification.from_pretrained("mlm", num_labels=2)
tokenizer = AutoTokenizer.from_pretrained("mlm")

## 11.4 命名实体识别

在本节中，我们将深入探讨如何为命名实体识别(NER)任务微调预训练的BERT模型。与对整个文档进行分类不同，这一过程能够对单个词元或单词进行细粒度分类（如人名、地点等）​。当处理敏感数据时，这对去标识化与匿名化任务尤为实用。

命名实体识别与本章开头讨论的文档分类任务有相似之处，关键的区别体现在数据预处理和分类方式上。由于我们的目标是对单个单词而非整个文档进行分类，在数据预处理阶段必须充分考虑这种细粒度特征。图11-18直观地展示了这种词级分类方法。

<div align="center">
    <img src="./picture/11-18.jpg">
    <p>图11-18：微调后的BERT模型能够识别人名和地点等命名实体</p>
</div>

微调预训练的BERT模型所采用的架构与文档分类任务中的架构相似，但分类机制存在本质差异。此时模型不再依赖词元嵌入的聚合或池化操作，而是直接对序列中的每个词元进行预测。需要特别说明的是，我们的词级分类任务并非作用于完整单词，而是针对构成单词的各个词元进行分类。图11-19清晰地呈现了这种词元级分类机制。

<div align="center">
    <img src="./picture/11-19.jpg">
    <p>图11-19：在微调过程中，BERT模型执行的是词元级分类而非文档级或词级分类</p>
</div>

### 11.4.1 数据准备


本示例采用英文版的CoNLL-2003数据集。该数据集涵盖人名、组织、地点、其他实体及非实体等类别，共包含约14000个训练样本。

In [ ]:
# 用于命名实体识别的CoNLL-2003数据集
dataset = load_dataset("conll2003", trust_remote_code=True)

在研究该示例数据集的过程中，我们还发现了其他一些值得分享的数据集。wnut_17数据集聚焦于新兴和罕见实体的识别任务，这类实体往往更难以识别。此外，tner/mit_movie_trivia和tner/mit_restaurant数据集也颇具研究价值。tner/mit_movie_trivia用于识别演员、情节与配乐等实体，tner/mit_restaurant则专注于设施、菜品及菜系等实体的识别。

我们通过具体示例来观察数据组织结构：

In [ ]:
example = dataset["train"][848]
example

该数据集为句子中的每个单词提供了标签。这些标签存储于ner_tags字段中，对应以下实体类型：

In [ ]:
label2id = {
    "O": 0, "B-PER": 1, "I-PER": 2, "B-ORG": 3, "I-ORG": 4,
    "B-LOC": 5, "I-LOC": 6, "B-MISC": 7, "I-MISC": 8
}
id2label = {index: label for label, index in label2id.items()}
label2id

这些实体对应特定类别：人名(PER)、组织(ORG)、地点(LOC)、其他实体(MISC)和非实体(O)。注意，这些实体的前缀为B（表示起始）或I（表示内部）​。当两个连续的词元属于同一个短语时，起始词元使用B标记，后续词元则用I标记，表明它们属于同一个实体，而不是独立存在。

图11-20直观地展示了这一过程。由于Dean被标记为短语的开头，Palmer被标记为短语的一部分（后面没有其他标记，代表Palmer同时是短语的结尾）​，因此我们可以判定DeanPalmer是一个完整的人名，Dean和Palmer并不是两个独立的人名。

<div align="center">
    <img src="./picture/11-20.jpg">
    <p>图11-20：通过用相同的实体标记短语的开头和结尾，实现完整短语的实体识别</p>
</div>


我们的数据已经过预处理并被拆分成单词，但尚未被转换为词元。为此，我们将使用本章始终沿用的预训练模型bert-base-cased的分词器对其进行进一步分词处理，从而得到词元。

In [ ]:
from transformers import AutoModelForTokenClassification
# 加载分词器
tokenizer = AutoTokenizer.from_pretrained("../models/bert-base-cased")
# 加载模型
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id
)

下面看一下分词器如何处理我们的示例：

In [ ]:
# 将单个词元拆分成子词元
token_ids = tokenizer(example["tokens"], is_split_into_words=True)["input_ids"]
sub_tokens = tokenizer.convert_ids_to_tokens(token_ids)
sub_tokens

正如我们在第2章和第3章中所学，分词器会添加[CLS]和[SEP]等特殊词元。值得注意的是，homer这个单词被进一步拆分为home和##r两个子词元。这引发了一个技术挑战：由于标注数据是基于完整单词而非子词元构建的，我们需要在分词过程中将原始标签与对应的子词元进行对齐。

以带B-PER标签（表示人名）的Maarten为例，当该词被分词器拆分为Ma、##arte和##n这三个子词元时，若简单地沿用原标签会导致错误——三个子词元都会被误判为独立的人名。当一个实体被拆分成多个词元时，首个词元应该带B（表示起始）标签，而接下来的词元应该带I（表示内部）标签。

以带B-PER标签（表示人名）的Maarten为例，当该词被分词器拆分为Ma、##arte和##n这三个子词元时，若简单地沿用原标签会导致错误——三个子词元都会被误判为独立的人名。当一个实体被拆分成多个词元时，首个词元应该带B（表示起始）标签，而接下来的词元应该带I（表示内部）标签。

<div align="center">
    <img src="./picture/11-21.jpg">
    <p>图11-21：分词输入标签的对齐过程</p>
</div>

为此，我们创建了align_labels函数，该函数在完成文本分词的同时动态调整标签，使其与子词元对齐。

In [ ]:
def align_labels(examples):
    token_ids = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )
    labels = examples["ner_tags"]
    updated_labels = []
    for index, label in enumerate(labels):
        # 将词元映射到它们各自的单词
        word_ids = token_ids.word_ids(batch_index=index)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # 新单词的开始
            if word_idx != previous_word_idx:
                previous_word_idx = word_idx
                updated_label = -100 if word_idx is None else label[word_idx]
                label_ids.append(updated_label)
            # 将特殊词元标记为-100
            elif word_idx is None:
                label_ids.append(-100)
            # 如果标签是B-XXX，我们将其改为I-XXX
            else:
                updated_label = label[word_idx]
                if updated_label % 2 == 1:
                    updated_label += 1
                label_ids.append(updated_label)
        updated_labels.append(label_ids)
    token_ids["labels"] = updated_labels
    return token_ids
tokenized = dataset.map(align_labels, batched=True)

通过观察示例可以发现，我们在[CLS]和[SEP]词元中额外添加了特殊标签(-100)。

In [ ]:
# 原始标签和更新后标签的区别
print(f"Original: {example["ner_tags"]}")
print(f"Updated: {tokenized["train"][848]["labels"]}")

现在，我们已经完成了词元化并与标签对齐，接下来需要确定评估指标的定义方式。这与之前的任务有所不同：在此前的场景中，每个文档仅需生成一个预测结果，而当前的词元化任务要求为每个词元生成相应的预测。

为此，我们将使用Hugging Face的evaluate工具包构建一个compute_metrics函数，该函数能够在词元粒度上评估模型性能。这种细粒度的评估机制将帮助我们更精确地分析模型对文本结构的理解能力。

In [ ]:
import evaluate
# 加载序列评估器
seqeval = evaluate.load("seqeval")
def compute_metrics(eval_pred):
    # 生成预测结果
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=2)
    true_predictions = []
    true_labels = []
    # 文档级循环
    for prediction, label in zip(predictions, labels):
      # 词元级循环
      for token_prediction, token_label in zip(prediction, label):
        # 忽略特殊词元
        if token_label != -100:
          true_predictions.append([id2label[token_prediction]])
          true_labels.append([id2label[token_label]])
    results = seqeval.compute(
    predictions=true_predictions, references=true_labels
)
    return {"f1": results["overall_f1"]}

### 11.4.2 命名实体识别的微调

我们即将完成准备工作。与DataCollatorWithPadding不同，此处需要一个能够处理词元级分类任务的数据整理器——DataCollatorForTokenClassification。

In [ ]:
from transformers import DataCollatorForTokenClassification
# 词元分类数据整理器
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

至此，我们完成了模型的加载工作，后续步骤与本章先前介绍的训练流程基本一致。我们需要通过定义包含特定参数的训练器来完成调优并实例化一个Trainer对象。

In [ ]:
# 用于参数调优的训练参数
training_args = TrainingArguments(
   "model",
   learning_rate=2e-5,
   per_device_train_batch_size=16,
   per_device_eval_batch_size=16,
   num_train_epochs=1,
   weight_decay=0.01,
   save_strategy="epoch",
   report_to="none"
)
# 初始化训练器
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()

接下来，我们对新构建的模型进行评估：

In [ ]:
# 在测试数据上评估模型
trainer.evaluate()

最后，我们保存训练好的模型并将其应用于推理流程。通过这种方式，我们既可以调阅特定数据、人工检视推理过程的中间结果，又能系统地验证输出内容是否符合预期。

In [ ]:
from transformers import pipeline
# 保存我们的微调模型
trainer.save_model("ner_model")
# 在微调模型上运行推理
token_classifier = pipeline(
    "token-classification",
    model="ner_model",
)
token_classifier("My name is Maarten.")

在句子"My name is Maarten."中，Maarten及其子词元被正确地识别为人名。

## 11.5 小结

在本章中，我们探讨了在特定分类任务上微调预训练表示模型的若干方法。首先，我们展示了如何对预训练的BERT模型进行微调，并通过冻结其架构的特定层来扩展应用场景。

我们尝试使用了名为SetFit的少样本分类技术，该技术利用有限的标注数据，同时微调预训练的嵌入模型和分类头。仅使用少量标注数据点，该模型就展现出与我们在前几章中所探讨的模型相似的性能表现。

随后，我们深入研究了继续预训练。我们以预训练的BERT模型为起点，使用不同的数据对其继续进行训练。继续预训练的底层机制——掩码语言建模，不仅用于创建表示模型，也可用于对模型进行继续预训练。

最后，我们探讨了命名实体识别任务，该任务需要从非结构化文本中识别特定实体（如人名、地名等）​。与之前的分类任务不同，此类分类作用于单词层面而非文档层面。

在第12章中，我们将继续探索语言模型的微调，重点转向生成模型。我们将分两个阶段展开：首先微调模型，使其准确执行指令；然后调整模型，使其输出符合人类偏好。